In [4]:
import pandas as pd
import os
from collections import deque
import time
import folium
from datetime import datetime
import webbrowser
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree
from geopy.distance import geodesic
import openrouteservice

class MultimodalTransitRouter:
    def __init__(self):
        self.api_key = '5b3ce3597851110001cf6248a6b7c97bb850491794bb504b30e2f2f7'
        start_time = time.time()
        print("Starting router initialization")
        
        self.load_transit_data()
        self._build_network()
        self._build_shape_cache()
        print(f"\nTotal initialization took {time.time() - start_time:.2f} seconds")

    def load_transit_data(self):
        """Load both bus and train GTFS data"""
        bus_dir = os.path.join('..', 'mpt_data', 'bus')
        train_dir = os.path.join('..', 'mpt_data', 'train')
        
        print("\nLoading GTFS files")
        file_load_start = time.time()
        
        # Load and combine bus and train data
        self.stops = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stops.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'stops.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.routes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'routes.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'routes.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.trips = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'trips.txt')),
            pd.read_csv(os.path.join(train_dir, 'trips.txt'))
        ], ignore_index=True)
        
        self.stop_times = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stop_times.txt')),
            pd.read_csv(os.path.join(train_dir, 'stop_times.txt'))
        ], ignore_index=True)
        
        self.shapes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'shapes.txt')),
            pd.read_csv(os.path.join(train_dir, 'shapes.txt'))
        ], ignore_index=True)
        
        print(f"File loading took {time.time() - file_load_start:.2f} seconds")
        
        # Normalize stop names and create search index
        self.stops['normalized_stop_name'] = self.stops['stop_name'].str.lower()

    def _build_network(self):
        """Build transit network and KDTree for spatial queries"""
        merge_start = time.time()
        
        # Build transit connections
        stop_route_data = self.stop_times.merge(
            self.trips[['trip_id', 'route_id']], 
            on='trip_id'
        )[['stop_id', 'route_id']]
        
        print(f"Merging dataframes took {time.time() - merge_start:.2f} seconds")
        
        mapping_start = time.time()
        self.stop_routes = {
            stop_id: set(group['route_id']) 
            for stop_id, group in stop_route_data.groupby('stop_id')
        }
        
        self.route_stops = {
            route_id: set(group['stop_id']) 
            for route_id, group in stop_route_data.groupby('route_id')
        }
        
        # Build KDTree for spatial queries
        self.kdtree = KDTree(self.stops[["stop_lat", "stop_lon"]].values)
        
        print(f"Building mappings took {time.time() - mapping_start:.2f} seconds")

    def _build_shape_cache(self):
        """Build cache of shape points for all routes"""
        print("Building shape cache")
        shape_start = time.time()
        
        # Get shape_id for each route using trips table
        route_shapes = {}
        for _, trip in self.trips.iterrows():
            route_shapes[trip['route_id']] = trip['shape_id']
        
        # Create cache of shape points
        self.route_shapes = {}
        for route_id, shape_id in route_shapes.items():
            shape_points = (
                self.shapes[self.shapes['shape_id'] == shape_id]
                .sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]
                .values.tolist()
            )
            self.route_shapes[route_id] = shape_points
        
        print(f"Shape cache built in {time.time() - shape_start:.2f} seconds")

    def _get_route_segment(self, route_id, start_stop_id, end_stop_id):
        """Get shape points for a segment of a route between two stops"""
        if route_id not in self.route_shapes:
            return []
            
        # Get stop coordinates
        start_stop = self.stops[self.stops['stop_id'] == start_stop_id].iloc[0]
        end_stop = self.stops[self.stops['stop_id'] == end_stop_id].iloc[0]
        start_coord = [start_stop['stop_lat'], start_stop['stop_lon']]
        end_coord = [end_stop['stop_lat'], end_stop['stop_lon']]
        
        # Get full shape for the route
        shape_points = self.route_shapes[route_id]
        
        # Find closest points on shape to our stops
        start_idx = min(range(len(shape_points)), 
                       key=lambda i: ((shape_points[i][0] - start_coord[0])**2 + 
                                    (shape_points[i][1] - start_coord[1])**2))
        end_idx = min(range(len(shape_points)), 
                     key=lambda i: ((shape_points[i][0] - end_coord[0])**2 + 
                                  (shape_points[i][1] - end_coord[1])**2))
        
        # Return points in correct order
        if start_idx <= end_idx:
            return shape_points[start_idx:end_idx + 1]
        else:
            return shape_points[end_idx:start_idx + 1][::-1]

    def find_nearest_stops(self, lat, lon, radius=500, limit=3):
        """Find stops within radius meters of the given coordinates"""
        # Query KDTree for nearest neighbors
        distances, indices = self.kdtree.query([lat, lon], k=10)  # Get more than we need
        
        nearby_stops = []
        for idx in indices:
            stop = self.stops.iloc[idx]
            dist = geodesic((lat, lon), (stop["stop_lat"], stop["stop_lon"])).meters
            if dist <= radius:
                nearby_stops.append((stop, dist))
                
        # Sort by distance and return top 'limit' stops
        return sorted(nearby_stops, key=lambda x: x[1])[:limit]

    def get_walking_route(self, from_lat, from_lon, to_lat, to_lon):
        """Get walking route using OpenRouteService"""
        client = openrouteservice.Client(key=self.api_key)
        coords = [[from_lon, from_lat], [to_lon, to_lat]]
        return client.directions(coordinates=coords, profile='foot-walking', format='geojson')

    def _find_multimodal_route(self, start_stops, end_stops):
        """Find route including walking between transit modes"""
        queue = deque([])
        visited = set()
        
        # Initialize queue with start stops
        for start_stop, start_dist in start_stops:
            queue.append((start_stop['stop_id'], [start_stop['stop_id']], [], start_dist))
            visited.add(start_stop['stop_id'])
        
        while queue:
            current_stop_id, path, transfers, total_dist = queue.popleft()
            
            # Check if we've reached a destination stop
            if any(current_stop_id == end_stop['stop_id'] for end_stop, _ in end_stops):
                return path, transfers
            
            # Get all routes from current stop
            routes = self.stop_routes.get(current_stop_id, set())
            
            # Try each route
            for route_id in routes:
                route_info = self.routes[self.routes['route_id'] == route_id].iloc[0]
                for next_stop_id in self.route_stops[route_id]:
                    if next_stop_id not in visited:
                        visited.add(next_stop_id)
                        
                        # Add transit segment
                        next_stop = self.stops[self.stops['stop_id'] == next_stop_id].iloc[0]
                        current_stop = self.stops[self.stops['stop_id'] == current_stop_id].iloc[0]
                        
                        new_transfers = transfers + [{
                            'type': 'transit',
                            'mode': route_info['mode'],
                            'route': route_info['route_short_name'],
                            'route_id': route_id,
                            'from_stop': current_stop_id,
                            'to_stop': next_stop_id,
                            'from_lat': current_stop['stop_lat'],
                            'from_lon': current_stop['stop_lon'],
                            'to_lat': next_stop['stop_lat'],
                            'to_lon': next_stop['stop_lon']
                        }]
                        
                        queue.append((next_stop_id, path + [next_stop_id], new_transfers, total_dist))
            
            # Try walking to nearby stops of different modes
            current_stop = self.stops[self.stops['stop_id'] == current_stop_id].iloc[0]
            nearby = self.find_nearest_stops(current_stop['stop_lat'], current_stop['stop_lon'])
            
            for next_stop, walk_dist in nearby:
                if next_stop['stop_id'] not in visited and next_stop['mode'] != current_stop['mode']:
                    visited.add(next_stop['stop_id'])
                    
                    # Add walking segment
                    new_transfers = transfers + [{
                        'type': 'walking',
                        'from_lat': current_stop['stop_lat'],
                        'from_lon': current_stop['stop_lon'],
                        'to_lat': next_stop['stop_lat'],
                        'to_lon': next_stop['stop_lon'],
                        'distance': walk_dist
                    }]
                    
                    queue.append((next_stop['stop_id'], path + [next_stop['stop_id']], new_transfers, total_dist + walk_dist))
        
        return None, None

    def visualize_route(self, start_coords, end_coords, path, transfers):
        """Visualize the complete route including walking segments"""
        m = folium.Map(location=[start_coords[0], start_coords[1]], zoom_start=13)
        
        # Colors for different modes
        colors = {
            'bus': {
                'blue': '#0066CC',
                'red': '#CC0000',
                'green': '#009933',
                'purple': '#660099',
                'orange': '#FF6600',
            },
            'train': {
                'blue': '#000066',
                'red': '#660000',
                'green': '#006600',
                'purple': '#330066',
                'orange': '#CC3300',
            }
        }
        
        # Add markers for start and end
        folium.Marker(
            [start_coords[0], start_coords[1]],
            popup='Start',
            icon=folium.Icon(color='green')
        ).add_to(m)
        
        folium.Marker(
            [end_coords[0], end_coords[1]],
            popup='End',
            icon=folium.Icon(color='red')
        ).add_to(m)
        
        # Track used colors for each mode
        used_colors = {'bus': 0, 'train': 0}
        
        # Draw each segment
        for transfer in transfers:
            if transfer['type'] == 'walking':
                # Get and draw walking route
                walking_route = self.get_walking_route(
                    transfer['from_lat'], transfer['from_lon'],
                    transfer['to_lat'], transfer['to_lon']
                )
                
                folium.GeoJson(
                    walking_route,
                    style_function=lambda x: {'color': '#00FF00', 'weight': 3, 'opacity': 0.7},
                    popup=f'Walking ({transfer["distance"]:.0f}m)'
                ).add_to(m)
                
            elif transfer['type'] == 'transit':
                # Get actual route path using shape points
                segment_points = self._get_route_segment(
                    transfer['route_id'],
                    transfer['from_stop'],
                    transfer['to_stop']
                )
                
                # Choose color based on mode
                mode = transfer['mode']
                color_idx = used_colors[mode]
                color = list(colors[mode].values())[color_idx % len(colors[mode])]
                used_colors[mode] += 1
                
                # Draw the route line using shape points
                folium.PolyLine(
                    locations=segment_points,
                    weight=4,
                    color=color,
                    popup=f"{mode.title()} {transfer['route']}",
                    opacity=0.8
                ).add_to(m)
                
                # Add route label at midpoint
                if len(segment_points) >= 2:
                    mid_idx = len(segment_points) // 2
                    mid_point = segment_points[mid_idx]
                    folium.DivIcon(
                        html=f'<div style="background-color: {color}; color: white; padding: 3px 6px; border-radius: 3px; font-weight: bold;">{mode.title()} {transfer["route"]}</div>',
                        icon_size=(70, 20),
                        icon_anchor=(35, 10)
                    ).add_to(folium.Marker(mid_point).add_to(m))
        
        # Add markers for transit stops
        for i, stop_id in enumerate(path):
            stop = self.stops[self.stops['stop_id'] == stop_id].iloc[0]
            
            if i == 0:
                icon_color = 'green'
                prefix = 'Start'
            elif i == len(path) - 1:
                icon_color = 'red'
                prefix = 'End'
            else:
                icon_color = 'blue'
                prefix = 'Transfer'
            
            folium.Marker(
                [stop['stop_lat'], stop['stop_lon']],
                popup=f"{prefix}: {stop['stop_name']}",
                icon=folium.Icon(color=icon_color, icon='info-sign')
            ).add_to(m)
        
# Save map
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        output_dir = os.path.join('..', 'mpt_data', 'maps')
        os.makedirs(output_dir, exist_ok=True)
        map_file = os.path.join(output_dir, f'multimodal_route_{timestamp}.html')
        m.save(map_file)
        return map_file

    def find_route(self, start_location, end_location):
        """Find and visualize a route between two locations"""
        # Geocode locations
        geolocator = Nominatim(user_agent="multimodal_router")
        start_loc = geolocator.geocode(start_location + ", Melbourne", exactly_one=True)
        end_loc = geolocator.geocode(end_location + ", Melbourne", exactly_one=True)
        
        if not start_loc or not end_loc:
            return "Could not geocode one or both locations"
        
        # Find nearest stops to start and end
        start_stops = self.find_nearest_stops(start_loc.latitude, start_loc.longitude)
        end_stops = self.find_nearest_stops(end_loc.latitude, end_loc.longitude)
        
        if not start_stops or not end_stops:
            return "No transit stops found near one or both locations"
        
        # Find route
        path, transfers = self._find_multimodal_route(start_stops, end_stops)
        
        if not path:
            return "No route found"
        
        # Add initial and final walking segments
        try:
            first_stop = self.stops[self.stops['stop_id'] == path[0]].iloc[0]
            last_stop = self.stops[self.stops['stop_id'] == path[-1]].iloc[0]
            
            # Initial walking segment
            initial_walk_dist = geodesic(
                (start_loc.latitude, start_loc.longitude),
                (first_stop['stop_lat'], first_stop['stop_lon'])
            ).meters
            
            # Final walking segment
            final_walk_dist = geodesic(
                (last_stop['stop_lat'], last_stop['stop_lon']),
                (end_loc.latitude, end_loc.longitude)
            ).meters
            
            # Add walking segments in consistent format
            transfers = [{
                'type': 'walking',
                'from_lat': start_loc.latitude,
                'from_lon': start_loc.longitude,
                'to_lat': first_stop['stop_lat'],
                'to_lon': first_stop['stop_lon'],
                'distance': initial_walk_dist,
                'description': f'Walk to {first_stop["stop_name"]}'
            }] + transfers + [{
                'type': 'walking',
                'from_lat': last_stop['stop_lat'],
                'from_lon': last_stop['stop_lon'],
                'to_lat': end_loc.latitude,
                'to_lon': end_loc.longitude,
                'distance': final_walk_dist,
                'description': f'Walk from {last_stop["stop_name"]} to destination'
            }]
            
        except Exception as e:
            print(f"Warning: Could not generate walking segments: {e}")
        
        # Visualize route
        map_file = self.visualize_route(
            (start_loc.latitude, start_loc.longitude),
            (end_loc.latitude, end_loc.longitude),
            path,
            transfers
        )
        
        # Generate text directions
        directions = ["Route found:"]
        total_walking = 0
        total_transfers = 0
        
        for transfer in transfers:
            if transfer['type'] == 'walking':
                if 'description' in transfer:
                    # Initial or final walking segment
                    directions.append(f"{transfer['description']} ({transfer['distance']:.0f}m)")
                else:
                    # Walking between stops
                    directions.append(f"Walk {transfer['distance']:.0f}m")
                total_walking += transfer['distance']
            else:
                # Transit segment
                directions.append(f"Take {transfer['mode']} {transfer['route']} to {self.stops[self.stops['stop_id'] == transfer['to_stop']].iloc[0]['stop_name']}")
                total_transfers += 1
        
        directions.append(f"\nTotal transfers: {total_transfers}")
        directions.append(f"Total walking distance: {total_walking:.0f}m")
        
        return "\n".join(directions), map_file

if __name__ == "__main__":
    router = MultimodalTransitRouter()
    
    print("\nPlease enter your start and end locations to find a route.\n")
    start = input("Enter starting location: ")
    end = input("Enter destination: ")
    
    result = router.find_route(start, end)
    if isinstance(result, tuple):
        directions, map_file = result
        print("\nDirections:")
        print(directions)
        print("\nMap saved to:", map_file)
        webbrowser.open('file://' + os.path.abspath(map_file))
    else:
        print("\nError:", result)

Starting router initialization

Loading GTFS files
File loading took 3.29 seconds
Merging dataframes took 0.59 seconds
Building mappings took 1.04 seconds
Building shape cache
Shape cache built in 7.91 seconds

Total initialization took 12.84 seconds

Please enter your start and end locations to find a route.



Enter starting location:  Essendon
Enter destination:  Prahran



Directions:
Route found:
Walk to 44-Thorn St/Mt Alexander Rd (Essendon) (234m)
Take bus 59 to 15-Murphy St/Flemington Rd (North Melbourne)
Take bus 58 to 22-Toorak Rd/St Kilda Rd (Melbourne City)
Take bus 6 to 30-Prahran Station/High St (Windsor)
Walk from 30-Prahran Station/High St (Windsor) to destination (169m)

Total transfers: 3
Total walking distance: 403m

Map saved to: ..\mpt_data\maps\multimodal_route_2024-12-10_04-28.html


In [10]:
import pandas as pd
import os
from collections import deque
import time
import folium
from datetime import datetime
import webbrowser
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree
from geopy.distance import geodesic
import openrouteservice
import pickle

class MultimodalTransitRouter:
    def __init__(self):
        self.api_key = '5b3ce3597851110001cf6248a6b7c97bb850491794bb504b30e2f2f7'
        start_time = time.time()
        print("Starting router initialization")
        
        self.load_transit_data()
        self._build_network()
        self._load_or_build_shape_cache()
        print(f"\nTotal initialization took {time.time() - start_time:.2f} seconds")

    def load_transit_data(self):
        """Load both bus and train GTFS data"""
        bus_dir = os.path.join('..', 'mpt_data', 'bus')
        train_dir = os.path.join('..', 'mpt_data', 'train')
        
        print("\nLoading GTFS files")
        file_load_start = time.time()
        
        # Load and combine bus and train data
        self.stops = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stops.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'stops.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.routes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'routes.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'routes.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.trips = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'trips.txt')),
            pd.read_csv(os.path.join(train_dir, 'trips.txt'))
        ], ignore_index=True)
        
        self.stop_times = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stop_times.txt')),
            pd.read_csv(os.path.join(train_dir, 'stop_times.txt'))
        ], ignore_index=True)
        
        self.shapes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'shapes.txt')),
            pd.read_csv(os.path.join(train_dir, 'shapes.txt'))
        ], ignore_index=True)
        
        print(f"File loading took {time.time() - file_load_start:.2f} seconds")
        
        # Normalize stop names and create search index
        self.stops['normalized_stop_name'] = self.stops['stop_name'].str.lower()

    def _build_network(self):
        """Build transit network and KDTree for spatial queries"""
        merge_start = time.time()
        
        # Build transit connections
        stop_route_data = self.stop_times.merge(
            self.trips[['trip_id', 'route_id']], 
            on='trip_id'
        )[['stop_id', 'route_id']]
        
        print(f"Merging dataframes took {time.time() - merge_start:.2f} seconds")
        
        mapping_start = time.time()
        self.stop_routes = {
            stop_id: set(group['route_id']) 
            for stop_id, group in stop_route_data.groupby('stop_id')
        }
        
        self.route_stops = {
            route_id: set(group['stop_id']) 
            for route_id, group in stop_route_data.groupby('route_id')
        }
        
        # Build KDTree for spatial queries
        self.kdtree = KDTree(self.stops[["stop_lat", "stop_lon"]].values)
        
        print(f"Building mappings took {time.time() - mapping_start:.2f} seconds")

    def _load_or_build_shape_cache(self):
        """Load shape cache from disk if it exists, otherwise build it"""
        cache_dir = os.path.join('..', 'mpt_data', 'cache')
        os.makedirs(cache_dir, exist_ok=True)
        cache_file = os.path.join(cache_dir, 'shape_cache.pkl')
        
        # Try to load existing cache
        if os.path.exists(cache_file):
            print("Loading shape cache from disk")
            cache_start = time.time()
            try:
                with open(cache_file, 'rb') as f:
                    self.route_shapes = pickle.load(f)
                print(f"Cache loaded in {time.time() - cache_start:.2f} seconds")
                return
            except Exception as e:
                print(f"Error loading cache: {e}")
        
        # Build cache if we get here
        print("Building shape cache")
        shape_start = time.time()
        
        # Get shape_id for each route using trips table
        route_shapes = {}
        for _, trip in self.trips.iterrows():
            route_shapes[trip['route_id']] = trip['shape_id']
        
        # Create cache of shape points
        self.route_shapes = {}
        for route_id, shape_id in route_shapes.items():
            shape_points = (
                self.shapes[self.shapes['shape_id'] == shape_id]
                .sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]
                .values.tolist()
            )
            self.route_shapes[route_id] = shape_points
        
        # Save cache
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump(self.route_shapes, f)
            print("Cache saved to disk")
        except Exception as e:
            print(f"Error saving cache: {e}")
        
        print(f"Shape cache built in {time.time() - shape_start:.2f} seconds")

    def _get_route_segment(self, route_id, start_stop_id, end_stop_id):
        """Get shape points for a segment of a route between two stops"""
        if route_id not in self.route_shapes:
            return []
            
        # Get stop coordinates
        start_stop = self.stops[self.stops['stop_id'] == start_stop_id].iloc[0]
        end_stop = self.stops[self.stops['stop_id'] == end_stop_id].iloc[0]
        start_coord = [start_stop['stop_lat'], start_stop['stop_lon']]
        end_coord = [end_stop['stop_lat'], end_stop['stop_lon']]
        
        # Get full shape for the route
        shape_points = self.route_shapes[route_id]
        
        # Find closest points on shape to our stops
        start_idx = min(range(len(shape_points)), 
                       key=lambda i: ((shape_points[i][0] - start_coord[0])**2 + 
                                    (shape_points[i][1] - start_coord[1])**2))
        end_idx = min(range(len(shape_points)), 
                     key=lambda i: ((shape_points[i][0] - end_coord[0])**2 + 
                                  (shape_points[i][1] - end_coord[1])**2))
        
        # Return points in correct order
        if start_idx <= end_idx:
            return shape_points[start_idx:end_idx + 1]
        else:
            return shape_points[end_idx:start_idx + 1][::-1]

    def find_nearest_stops(self, lat, lon, radius=500, limit=3):
        """Find stops within radius meters of the given coordinates"""
        # Query KDTree for nearest neighbors
        distances, indices = self.kdtree.query([lat, lon], k=10)  # Get more than we need
        
        nearby_stops = []
        for idx in indices:
            stop = self.stops.iloc[idx]
            dist = geodesic((lat, lon), (stop["stop_lat"], stop["stop_lon"])).meters
            if dist <= radius:
                nearby_stops.append((stop, dist))
                
        # Sort by distance and return top 'limit' stops
        return sorted(nearby_stops, key=lambda x: x[1])[:limit]

    def get_walking_route(self, from_lat, from_lon, to_lat, to_lon):
        """Get walking route using OpenRouteService"""
        client = openrouteservice.Client(key=self.api_key)
        coords = [[from_lon, from_lat], [to_lon, to_lat]]
        return client.directions(coordinates=coords, profile='foot-walking', format='geojson')

    def _find_multimodal_route(self, start_stops, end_stops):
        """Find route including walking between transit modes"""
        queue = deque([])
        visited = set()
        
        # Initialize queue with start stops
        for start_stop, start_dist in start_stops:
            queue.append((start_stop['stop_id'], [start_stop['stop_id']], [], start_dist))
            visited.add(start_stop['stop_id'])
        
        while queue:
            current_stop_id, path, transfers, total_dist = queue.popleft()
            
            # Check if we've reached a destination stop
            if any(current_stop_id == end_stop['stop_id'] for end_stop, _ in end_stops):
                return path, transfers
            
            # Get all routes from current stop
            routes = self.stop_routes.get(current_stop_id, set())
            
            # Try each route
            for route_id in routes:
                route_info = self.routes[self.routes['route_id'] == route_id].iloc[0]
                for next_stop_id in self.route_stops[route_id]:
                    if next_stop_id not in visited:
                        visited.add(next_stop_id)
                        
                        # Add transit segment
                        next_stop = self.stops[self.stops['stop_id'] == next_stop_id].iloc[0]
                        current_stop = self.stops[self.stops['stop_id'] == current_stop_id].iloc[0]
                        
                        new_transfers = transfers + [{
                            'type': 'transit',
                            'mode': route_info['mode'],
                            'route': route_info['route_short_name'],
                            'route_id': route_id,
                            'from_stop': current_stop_id,
                            'to_stop': next_stop_id,
                            'from_lat': current_stop['stop_lat'],
                            'from_lon': current_stop['stop_lon'],
                            'to_lat': next_stop['stop_lat'],
                            'to_lon': next_stop['stop_lon']
                        }]
                        
                        queue.append((next_stop_id, path + [next_stop_id], new_transfers, total_dist))
            
            # Try walking to nearby stops of different modes
            current_stop = self.stops[self.stops['stop_id'] == current_stop_id].iloc[0]
            nearby = self.find_nearest_stops(current_stop['stop_lat'], current_stop['stop_lon'])
            
            for next_stop, walk_dist in nearby:
                if next_stop['stop_id'] not in visited and next_stop['mode'] != current_stop['mode']:
                    visited.add(next_stop['stop_id'])
                    
                    # Add walking segment
                    new_transfers = transfers + [{
                        'type': 'walking',
                        'from_lat': current_stop['stop_lat'],
                        'from_lon': current_stop['stop_lon'],
                        'to_lat': next_stop['stop_lat'],
                        'to_lon': next_stop['stop_lon'],
                        'distance': walk_dist
                    }]
                    
                    queue.append((next_stop['stop_id'], path + [next_stop['stop_id']], new_transfers, total_dist + walk_dist))
        
        return None, None

    def visualize_route(self, start_coords, end_coords, path, transfers):
        """Visualize the complete route including walking segments"""
        m = folium.Map(location=[start_coords[0], start_coords[1]], zoom_start=13)
        
        # Colors for different modes
        colors = {
            'bus': {
                'blue': '#0066CC',
                'red': '#CC0000',
                'green': '#009933',
                'purple': '#660099',
                'orange': '#FF6600',
            },
            'train': {
                'blue': '#000066',
                'red': '#660000',
                'green': '#006600',
                'purple': '#330066',
                'orange': '#CC3300',
            }
        }
        
        # Add markers for start and end
        folium.Marker(
            [start_coords[0], start_coords[1]],
            popup='Start',
            icon=folium.Icon(color='green')
        ).add_to(m)
        
        folium.Marker(
            [end_coords[0], end_coords[1]],
            popup='End',
            icon=folium.Icon(color='red')
        ).add_to(m)
        
        # Track used colors for each mode
        used_colors = {'bus': 0, 'train': 0}
        
        # Draw each segment
        for transfer in transfers:
            if transfer['type'] == 'walking':
                # Get and draw walking route
                walking_route = self.get_walking_route(
                    transfer['from_lat'], transfer['from_lon'],
                    transfer['to_lat'], transfer['to_lon']
                )
                
                folium.GeoJson(
                    walking_route,
                    style_function=lambda x: {'color': '#00FF00', 'weight': 3, 'opacity': 0.7},
                    popup=f'Walking ({transfer["distance"]:.0f}m)'
                ).add_to(m)
                
            elif transfer['type'] == 'transit':
                # Get actual route path using shape points
                segment_points = self._get_route_segment(
                    transfer['route_id'],
                    transfer['from_stop'],
                    transfer['to_stop']
                )
                
                # Choose color based on mode
                mode = transfer['mode']
                color_idx = used_colors[mode]
                color = list(colors[mode].values())[color_idx % len(colors[mode])]
                used_colors[mode] += 1
                
                # Draw the route line using shape points
                folium.PolyLine(
                    locations=segment_points,
                    weight=4,
                    color=color,
                    popup=f"{mode.title()} {transfer['route']}",
                    opacity=0.8
                ).add_to(m)
                
                # Add route label at midpoint
                if len(segment_points) >= 2:
                    mid_idx = len(segment_points) // 2
                    mid_point = segment_points[mid_idx]
                    folium.DivIcon(
                        html=f'<div style="background-color: {color}; color: white; padding: 3px 6px; border-radius: 3px; font-weight: bold;">{mode.title()} {transfer["route"]}</div>',
                        icon_size=(70, 20),
                        icon_anchor=(35, 10)
                    ).add_to(folium.Marker(mid_point).add_to(m))
        
        # Add markers for transit stops
        for i, stop_id in enumerate(path):
            stop = self.stops[self.stops['stop_id'] == stop_id].iloc[0]
            
            if i == 0:
                icon_color = 'green'
                prefix = 'Start'
            elif i == len(path) - 1:
                icon_color = 'red'
                prefix = 'End'
            else:
                icon_color = 'blue'
                prefix = 'Transfer'
            
            folium.Marker(
                [stop['stop_lat'], stop['stop_lon']],
                popup=f"{prefix}: {stop['stop_name']}",
                icon=folium.Icon(color=icon_color, icon='info-sign')
            ).add_to(m)
        
        # Save map
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        output_dir = os.path.join('..', 'mpt_data', 'maps')
        os.makedirs(output_dir, exist_ok=True)
        map_file = os.path.join(output_dir, f'multimodal_route_{timestamp}.html')
        m.save(map_file)
        return map_file

    def find_route(self, start_location, end_location):
        """Find and visualize a route between two locations"""
        # Geocode locations
        geolocator = Nominatim(user_agent="multimodal_router")
        start_loc = geolocator.geocode(start_location + ", Melbourne", exactly_one=True)
        end_loc = geolocator.geocode(end_location + ", Melbourne", exactly_one=True)
        
        if not start_loc or not end_loc:
            return "Could not geocode one or both locations"
        
        # Find nearest stops to start and end
        start_stops = self.find_nearest_stops(start_loc.latitude, start_loc.longitude)
        end_stops = self.find_nearest_stops(end_loc.latitude, end_loc.longitude)
        
        if not start_stops or not end_stops:
            return "No transit stops found near one or both locations"
        
        # Find route
        path, transfers = self._find_multimodal_route(start_stops, end_stops)
        
        if not path:
            return "No route found"
        
        # Add initial and final walking segments
        try:
            first_stop = self.stops[self.stops['stop_id'] == path[0]].iloc[0]
            last_stop = self.stops[self.stops['stop_id'] == path[-1]].iloc[0]
            
            # Initial walking segment
            initial_walk_dist = geodesic(
                (start_loc.latitude, start_loc.longitude),
                (first_stop['stop_lat'], first_stop['stop_lon'])
            ).meters
            
            # Final walking segment
            final_walk_dist = geodesic(
                (last_stop['stop_lat'], last_stop['stop_lon']),
                (end_loc.latitude, end_loc.longitude)
            ).meters
            
            # Add walking segments in consistent format
            transfers = [{
                'type': 'walking',
                'from_lat': start_loc.latitude,
                'from_lon': start_loc.longitude,
                'to_lat': first_stop['stop_lat'],
                'to_lon': first_stop['stop_lon'],
                'distance': initial_walk_dist,
                'description': f'Walk to {first_stop["stop_name"]}'
            }] + transfers + [{
                'type': 'walking',
                'from_lat': last_stop['stop_lat'],
                'from_lon': last_stop['stop_lon'],
                'to_lat': end_loc.latitude,
                'to_lon': end_loc.longitude,
                'distance': final_walk_dist,
                'description': f'Walk from {last_stop["stop_name"]} to destination'
            }]
            
        except Exception as e:
            print(f"Warning: Could not generate walking segments: {e}")
        
        # Visualize route
        map_file = self.visualize_route(
            (start_loc.latitude, start_loc.longitude),
            (end_loc.latitude, end_loc.longitude),
            path,
            transfers
        )
        
        # Generate text directions
        directions = ["Route found:"]
        total_walking = 0
        total_transfers = 0
        
        for transfer in transfers:
            if transfer['type'] == 'walking':
                if 'description' in transfer:
                    # Initial or final walking segment
                    directions.append(f"{transfer['description']} ({transfer['distance']:.0f}m)")
                else:
                    # Walking between stops
                    directions.append(f"Walk {transfer['distance']:.0f}m")
                total_walking += transfer['distance']
            else:
                # Transit segment
                directions.append(f"Take {transfer['mode']} {transfer['route']} to {self.stops[self.stops['stop_id'] == transfer['to_stop']].iloc[0]['stop_name']}")
                total_transfers += 1
        
        directions.append(f"\nTotal transfers: {total_transfers}")
        directions.append(f"Total walking distance: {total_walking:.0f}m")
        
        return "\n".join(directions), map_file

if __name__ == "__main__":
    router = MultimodalTransitRouter()
    
    print("\nPlease enter your start and end locations to find a route.\n")
    start = input("Enter starting location: ")
    end = input("Enter destination: ")
    
    result = router.find_route(start, end)
    if isinstance(result, tuple):
        directions, map_file = result
        print("\nDirections:")
        print(directions)
        print("\nMap saved to:", map_file)
        webbrowser.open('file://' + os.path.abspath(map_file))
    else:
        print("\nError:", result)

Starting router initialization

Loading GTFS files
File loading took 2.64 seconds
Merging dataframes took 0.45 seconds
Building mappings took 0.92 seconds
Loading shape cache from disk
Cache loaded in 0.01 seconds

Total initialization took 4.01 seconds

Please enter your start and end locations to find a route.



Enter starting location:  Essendon
Enter destination:  Prahran



Directions:
Route found:
Walk to 44-Thorn St/Mt Alexander Rd (Essendon) (234m)
Take bus 59 to 15-Murphy St/Flemington Rd (North Melbourne)
Take bus 58 to 22-Toorak Rd/St Kilda Rd (Melbourne City)
Take bus 6 to 30-Prahran Station/High St (Windsor)
Walk from 30-Prahran Station/High St (Windsor) to destination (169m)

Total transfers: 3
Total walking distance: 403m

Map saved to: ..\mpt_data\maps\multimodal_route_2024-12-10_22-11.html


In [9]:
import pandas as pd
import os
from collections import deque
import time
import folium
from datetime import datetime, timedelta
import webbrowser
from geopy.geocoders import Nominatim
from scipy.spatial import KDTree
from geopy.distance import geodesic
import openrouteservice
import pickle
import pytz

class MultimodalTransitRouter:
    def __init__(self):
        self.api_key = '5b3ce3597851110001cf6248a6b7c97bb850491794bb504b30e2f2f7'
        start_time = time.time()
        print("Starting router initialization")
        
        self.load_transit_data()
        self._build_network()
        self._load_or_build_shape_cache()
        print(f"\nTotal initialization took {time.time() - start_time:.2f} seconds")

    def load_transit_data(self):
        """Load both bus and train GTFS data"""
        bus_dir = os.path.join('..', 'mpt_data', 'bus')
        train_dir = os.path.join('..', 'mpt_data', 'train')
        
        print("\nLoading GTFS files")
        file_load_start = time.time()
        
        # Load and combine bus and train data
        self.stops = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stops.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'stops.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.routes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'routes.txt')).assign(mode='bus'),
            pd.read_csv(os.path.join(train_dir, 'routes.txt')).assign(mode='train')
        ], ignore_index=True)
        
        self.trips = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'trips.txt')),
            pd.read_csv(os.path.join(train_dir, 'trips.txt'))
        ], ignore_index=True)
        
        self.stop_times = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'stop_times.txt')),
            pd.read_csv(os.path.join(train_dir, 'stop_times.txt'))
        ], ignore_index=True)
        
        self.shapes = pd.concat([
            pd.read_csv(os.path.join(bus_dir, 'shapes.txt')),
            pd.read_csv(os.path.join(train_dir, 'shapes.txt'))
        ], ignore_index=True)
        
        print(f"File loading took {time.time() - file_load_start:.2f} seconds")
        
        # Normalize stop names and create search index
        self.stops['normalized_stop_name'] = self.stops['stop_name'].str.lower()

    def _build_network(self):
        """Build transit network and KDTree for spatial queries"""
        merge_start = time.time()
        
        # Build transit connections
        stop_route_data = self.stop_times.merge(
            self.trips[['trip_id', 'route_id']], 
            on='trip_id'
        )[['stop_id', 'route_id']]
        
        print(f"Merging dataframes took {time.time() - merge_start:.2f} seconds")
        
        mapping_start = time.time()
        self.stop_routes = {
            stop_id: set(group['route_id']) 
            for stop_id, group in stop_route_data.groupby('stop_id')
        }
        
        self.route_stops = {
            route_id: set(group['stop_id']) 
            for route_id, group in stop_route_data.groupby('route_id')
        }
        
        # Build KDTree for spatial queries
        self.kdtree = KDTree(self.stops[["stop_lat", "stop_lon"]].values)
        
        print(f"Building mappings took {time.time() - mapping_start:.2f} seconds")

    def _load_or_build_shape_cache(self):
        """Load shape cache from disk if it exists, otherwise build it"""
        cache_dir = os.path.join('..', 'mpt_data', 'cache')
        os.makedirs(cache_dir, exist_ok=True)
        cache_file = os.path.join(cache_dir, 'shape_cache.pkl')
        
        # Try to load existing cache
        if os.path.exists(cache_file):
            print("Loading shape cache from disk")
            cache_start = time.time()
            try:
                with open(cache_file, 'rb') as f:
                    self.route_shapes = pickle.load(f)
                print(f"Cache loaded in {time.time() - cache_start:.2f} seconds")
                return
            except Exception as e:
                print(f"Error loading cache: {e}")
        
        # Build cache if we get here
        print("Building shape cache")
        shape_start = time.time()
        
        # Get shape_id for each route using trips table
        route_shapes = {}
        for _, trip in self.trips.iterrows():
            route_shapes[trip['route_id']] = trip['shape_id']
        
        # Create cache of shape points
        self.route_shapes = {}
        for route_id, shape_id in route_shapes.items():
            shape_points = (
                self.shapes[self.shapes['shape_id'] == shape_id]
                .sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]
                .values.tolist()
            )
            self.route_shapes[route_id] = shape_points
        
        # Save cache
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump(self.route_shapes, f)
            print("Cache saved to disk")
        except Exception as e:
            print(f"Error saving cache: {e}")
        
        print(f"Shape cache built in {time.time() - shape_start:.2f} seconds")

    def _get_route_segment(self, route_id, start_stop_id, end_stop_id):
        """Get shape points for a segment of a route between two stops"""
        if route_id not in self.route_shapes:
            return []
            
        # Get stop coordinates
        start_stop = self.stops[self.stops['stop_id'] == start_stop_id].iloc[0]
        end_stop = self.stops[self.stops['stop_id'] == end_stop_id].iloc[0]
        start_coord = [start_stop['stop_lat'], start_stop['stop_lon']]
        end_coord = [end_stop['stop_lat'], end_stop['stop_lon']]
        
        # Get full shape for the route
        shape_points = self.route_shapes[route_id]
        
        # Find closest points on shape to our stops
        start_idx = min(range(len(shape_points)), 
                       key=lambda i: ((shape_points[i][0] - start_coord[0])**2 + 
                                    (shape_points[i][1] - start_coord[1])**2))
        end_idx = min(range(len(shape_points)), 
                     key=lambda i: ((shape_points[i][0] - end_coord[0])**2 + 
                                  (shape_points[i][1] - end_coord[1])**2))
        
        # Return points in correct order
        if start_idx <= end_idx:
            return shape_points[start_idx:end_idx + 1]
        else:
            return shape_points[end_idx:start_idx + 1][::-1]

    def find_nearest_stops(self, lat, lon, radius=500, limit=3):
        """Find stops within radius meters of the given coordinates"""
        # Query KDTree for nearest neighbors
        distances, indices = self.kdtree.query([lat, lon], k=10)  # Get more than we need
        
        nearby_stops = []
        for idx in indices:
            stop = self.stops.iloc[idx]
            dist = geodesic((lat, lon), (stop["stop_lat"], stop["stop_lon"])).meters
            if dist <= radius:
                nearby_stops.append((stop, dist))
                
        # Sort by distance and return top 'limit' stops
        return sorted(nearby_stops, key=lambda x: x[1])[:limit]

    def get_walking_route(self, from_lat, from_lon, to_lat, to_lon):
        """Get walking route using OpenRouteService"""
        client = openrouteservice.Client(key=self.api_key)
        coords = [[from_lon, from_lat], [to_lon, to_lat]]
        return client.directions(coordinates=coords, profile='foot-walking', format='geojson')

    def get_walking_time(self, from_lat, from_lon, to_lat, to_lon):
        """Get actual walking time in seconds using OpenRouteService routing"""
        route = self.get_walking_route(from_lat, from_lon, to_lat, to_lon)
        return route['features'][0]['properties']['segments'][0]['duration']

    def _find_multimodal_route(self, start_stops, end_stops, departure_time=None):
        """Find route using real schedule data starting from current Melbourne time"""
        import heapq
        
        # Use current Melbourne time if no departure time specified
        melbourne_tz = pytz.timezone('Australia/Melbourne')
        if departure_time is None:
            departure_time = datetime.now(melbourne_tz)

        # Initialize priority queue and visited tracking
        pq = []  # Format: (total_time, stop_id, current_time, path, transfers)
        visited = {}  # track best time to each node

        def get_next_departure(route_id, stop_id, after_time):
            """Get next actual departure time after given time"""
            # Convert datetime to GTFS time format (handles >24h times)
            day_seconds = (after_time.hour * 3600 + 
                          after_time.minute * 60 + 
                          after_time.second)
            
            # Get all trips for this route
            route_trips = self.trips[self.trips['route_id'] == route_id]['trip_id']
            
            if route_trips.empty:
                return None
                
            # Get all stop times for these trips at this stop
            schedule = self.stop_times[
                (self.stop_times['stop_id'] == stop_id) & 
                (self.stop_times['trip_id'].isin(route_trips))
            ].copy()  # Create explicit copy to avoid SettingWithCopyWarning
            
            if schedule.empty:
                return None
                
            # Convert departure_time to seconds past midnight
            schedule.loc[:, 'seconds'] = schedule['departure_time'].apply(
                lambda x: sum(int(i) * m for i, m in zip(x.split(':'), [3600, 60, 1]))
            )
            
            # Find next departure
            next_departure = schedule[schedule['seconds'] > day_seconds]
            
            if not next_departure.empty:
                # Found a departure later today
                seconds = int(next_departure.iloc[0]['seconds'])  # Convert numpy.int64 to Python int
                
                # Handle times past midnight (>24 hours)
                days_to_add = int(seconds // (24 * 3600))  # Convert to Python int
                adjusted_seconds = int(seconds % (24 * 3600))  # Convert to Python int
                
                # Create new datetime for the departure
                dep_time = after_time + timedelta(days=days_to_add)
                dep_time = dep_time.replace(
                    hour=adjusted_seconds // 3600,
                    minute=(adjusted_seconds % 3600) // 60,
                    second=adjusted_seconds % 60
                )
                return dep_time
            else:
                # Look for first service next day
                if schedule['seconds'].empty:
                    return None
                first_departure = schedule.loc[schedule['seconds'].idxmin()]
                seconds = int(first_departure['seconds'])  # Convert numpy.int64 to Python int
                
                # Handle times past midnight for next day
                days_to_add = int(1 + (seconds // (24 * 3600)))  # Convert to Python int
                adjusted_seconds = int(seconds % (24 * 3600))  # Convert to Python int
                
                # Create datetime for tomorrow's first service
                next_day = after_time + timedelta(days=days_to_add)
                dep_time = next_day.replace(
                    hour=adjusted_seconds // 3600,
                    minute=(adjusted_seconds % 3600) // 60,
                    second=adjusted_seconds % 60
                )
                return dep_time

        def get_transit_time(route_id, from_stop_id, to_stop_id):
            """Get actual transit time between stops from schedule"""
            # Get all trips for this route
            trips = self.trips[self.trips['route_id'] == route_id]['trip_id']
            
            if trips.empty:
                return None
                
            # Get stop times for these stops on this route
            times = self.stop_times[
                (self.stop_times['trip_id'].isin(trips)) &
                (self.stop_times['stop_id'].isin([from_stop_id, to_stop_id]))
            ].copy()
            
            if times.empty:
                return None
                
            # Convert times to seconds
            times['seconds'] = times['arrival_time'].apply(
                lambda x: sum(int(i) * m for i, m in zip(x.split(':'), [3600, 60, 1]))
            )
            
            # Calculate transit time for each trip
            trip_times = []
            for trip_id in trips:
                trip_stops = times[times['trip_id'] == trip_id].sort_values('stop_sequence')
                if len(trip_stops) == 2:
                    trip_times.append(trip_stops.iloc[1]['seconds'] - trip_stops.iloc[0]['seconds'])
            
            if not trip_times:
                return None
                
            # Use median transit time to avoid outliers
            return sorted(trip_times)[len(trip_times)//2]

        # Initialize with start stops
        for start_stop, start_dist in start_stops:
            # Get walking time to first stop
            initial_walk_time = self.get_walking_time(
                start_stop['stop_lat'], start_stop['stop_lon'],
                start_stop['stop_lat'], start_stop['stop_lon']
            )
            arrival_time = departure_time + timedelta(seconds=initial_walk_time)
            
            initial_transfer = [{
                'type': 'walking',
                'from_lat': start_stop['stop_lat'],
                'from_lon': start_stop['stop_lon'],
                'to_lat': start_stop['stop_lat'],
                'to_lon': start_stop['stop_lon'],
                'distance': start_dist,
                'duration': initial_walk_time,
                'start_time': departure_time,
                'end_time': arrival_time
            }]
            
            heapq.heappush(pq, (
                initial_walk_time,
                start_stop['stop_id'],
                arrival_time,
                [start_stop['stop_id']],
                initial_transfer
            ))
            visited[start_stop['stop_id']] = initial_walk_time

        while pq:
            total_time, current_id, current_time, path, transfers = heapq.heappop(pq)
            
            if current_id in visited and visited[current_id] < total_time:
                continue
                
            if any(current_id == end_stop['stop_id'] for end_stop, _ in end_stops):
                return path, transfers
            
            current_stop = self.stops[self.stops['stop_id'] == current_id].iloc[0]
            
            # Try transit options
            routes = self.stop_routes.get(current_id, set())
            for route_id in routes:
                route_info = self.routes[self.routes['route_id'] == route_id].iloc[0]
                
                # Get next actual departure
                departure_time = get_next_departure(route_id, current_id, current_time)
                if departure_time is None:
                    continue
                    
                wait_time = (departure_time - current_time).total_seconds()
                
                for next_stop_id in self.route_stops[route_id]:
                    if next_stop_id == current_id:
                        continue
                        
                    transit_time = get_transit_time(route_id, current_id, next_stop_id)
                    if transit_time is None:
                        continue
                        
                    next_stop = self.stops[self.stops['stop_id'] == next_stop_id].iloc[0]
                    new_total_time = total_time + wait_time + transit_time
                    arrival_time = departure_time + timedelta(seconds=transit_time)
                    
                    if next_stop_id not in visited or new_total_time < visited[next_stop_id]:
                        visited[next_stop_id] = new_total_time
                        new_transfers = transfers + [{
                            'type': 'transit',
                            'mode': route_info['mode'],
                            'route': route_info['route_short_name'],
                            'route_id': route_id,
                            'from_stop': current_id,
                            'to_stop': next_stop_id,
                            'from_lat': current_stop['stop_lat'],
                            'from_lon': current_stop['stop_lon'],
                            'to_lat': next_stop['stop_lat'],
                            'to_lon': next_stop['stop_lon'],
                            'departure_time': departure_time,
                            'arrival_time': arrival_time,
                            'wait_time': wait_time,
                            'transit_time': transit_time
                        }]
                        
                        heapq.heappush(pq, (
                            new_total_time,
                            next_stop_id,
                            arrival_time,
                            path + [next_stop_id],
                            new_transfers
                        ))

            # Try walking transfers
            nearby = self.find_nearest_stops(current_stop['stop_lat'], current_stop['stop_lon'])
            for next_stop, _ in nearby:
                if next_stop['mode'] != current_stop['mode']:
                    walk_time = self.get_walking_time(
                        current_stop['stop_lat'], current_stop['stop_lon'],
                        next_stop['stop_lat'], next_stop['stop_lon']
                    )
                    
                    new_total_time = total_time + walk_time
                    arrival_time = current_time + timedelta(seconds=walk_time)
                    
                    if next_stop['stop_id'] not in visited or new_total_time < visited[next_stop['stop_id']]:
                        visited[next_stop['stop_id']] = new_total_time
                        new_transfers = transfers + [{
                            'type': 'walking',
                            'from_lat': current_stop['stop_lat'],
                            'from_lon': current_stop['stop_lon'],
                            'to_lat': next_stop['stop_lat'],
                            'to_lon': next_stop['stop_lon'],
                            'duration': walk_time,
                            'start_time': current_time,
                            'end_time': arrival_time
                        }]
                        
                        heapq.heappush(pq, (
                            new_total_time,
                            next_stop['stop_id'],
                            arrival_time,
                            path + [next_stop['stop_id']],
                            new_transfers
                        ))

        return None, None

    def visualize_route(self, start_coords, end_coords, path, transfers):
        """Visualize the complete route including walking segments and timing"""
        m = folium.Map(location=[start_coords[0], start_coords[1]], zoom_start=13)
        
        # Colors for different modes
        colors = {
            'bus': {
                'blue': '#0066CC',
                'red': '#CC0000',
                'green': '#009933',
                'purple': '#660099',
                'orange': '#FF6600',
            },
            'train': {
                'blue': '#000066',
                'red': '#660000',
                'green': '#006600',
                'purple': '#330066',
                'orange': '#CC3300',
            }
        }
        
        # Add markers for start and end
        folium.Marker(
            [start_coords[0], start_coords[1]],
            popup=f'Start: {transfers[0]["start_time"].strftime("%H:%M")}',
            icon=folium.Icon(color='green')
        ).add_to(m)
        
        folium.Marker(
            [end_coords[0], end_coords[1]],
            popup=f'End: {transfers[-1]["end_time"].strftime("%H:%M")}',
            icon=folium.Icon(color='red')
        ).add_to(m)
        
        # Track used colors for each mode
        used_colors = {'bus': 0, 'train': 0}
        
        # Draw each segment
        for transfer in transfers:
            if transfer['type'] == 'walking':
                # Get and draw walking route
                walking_route = self.get_walking_route(
                    transfer['from_lat'], transfer['from_lon'],
                    transfer['to_lat'], transfer['to_lon']
                )
                
                start_time = transfer['start_time'].strftime('%H:%M')
                end_time = transfer['end_time'].strftime('%H:%M')
                duration = transfer['duration'] / 60  # Convert to minutes
                
                folium.GeoJson(
                    walking_route,
                    style_function=lambda x: {'color': '#00FF00', 'weight': 3, 'opacity': 0.7},
                    popup=f'Walking ({duration:.0f}min)<br>{start_time} - {end_time}'
                ).add_to(m)
                
            elif transfer['type'] == 'transit':
                # Get actual route path using shape points
                segment_points = self._get_route_segment(
                    transfer['route_id'],
                    transfer['from_stop'],
                    transfer['to_stop']
                )
                
                # Choose color based on mode
                mode = transfer['mode']
                color_idx = used_colors[mode]
                color = list(colors[mode].values())[color_idx % len(colors[mode])]
                used_colors[mode] += 1
                
                # Format times
                departure = transfer['departure_time'].strftime('%H:%M')
                arrival = transfer['arrival_time'].strftime('%H:%M')
                duration = transfer['transit_time'] / 60  # Convert to minutes
                
                # Draw the route line using shape points
                folium.PolyLine(
                    locations=segment_points,
                    weight=4,
                    color=color,
                    popup=f"{mode.title()} {transfer['route']}<br>{departure} - {arrival}<br>Duration: {duration:.0f}min",
                    opacity=0.8
                ).add_to(m)
                
                # Add route label at midpoint
                if len(segment_points) >= 2:
                    mid_idx = len(segment_points) // 2
                    mid_point = segment_points[mid_idx]
                    folium.DivIcon(
                        html=f'<div style="background-color: {color}; color: white; padding: 3px 6px; border-radius: 3px; font-weight: bold;">{mode.title()} {transfer["route"]}</div>',
                        icon_size=(70, 20),
                        icon_anchor=(35, 10)
                    ).add_to(folium.Marker(mid_point).add_to(m))
        
        # Add markers for transit stops with times
        for i, stop_id in enumerate(path):
            stop = self.stops[self.stops['stop_id'] == stop_id].iloc[0]
            
            # Find corresponding transfer for this stop
            stop_times = []
            for t in transfers:
                if t['type'] == 'transit':
                    if t['from_stop'] == stop_id:
                        stop_times.append(f"Dep: {t['departure_time'].strftime('%H:%M')}")
                    elif t['to_stop'] == stop_id:
                        stop_times.append(f"Arr: {t['arrival_time'].strftime('%H:%M')}")
            
            if i == 0:
                icon_color = 'green'
                prefix = 'Start'
            elif i == len(path) - 1:
                icon_color = 'red'
                prefix = 'End'
            else:
                icon_color = 'blue'
                prefix = 'Transfer'
            
            popup_text = f"{prefix}: {stop['stop_name']}"
            if stop_times:
                popup_text += f"<br>{'<br>'.join(stop_times)}"
            
            folium.Marker(
                [stop['stop_lat'], stop['stop_lon']],
                popup=popup_text,
                icon=folium.Icon(color=icon_color, icon='info-sign')
            ).add_to(m)
        
        # Save map
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        output_dir = os.path.join('..', 'mpt_data', 'maps')
        os.makedirs(output_dir, exist_ok=True)
        map_file = os.path.join(output_dir, f'multimodal_route_{timestamp}.html')
        m.save(map_file)
        return map_file

    def find_route(self, start_location, end_location, departure_time=None):
        """Find and visualize a route between two locations with timing details"""
        # Geocode locations
        geolocator = Nominatim(user_agent="multimodal_router")
        start_loc = geolocator.geocode(start_location + ", Melbourne", exactly_one=True)
        end_loc = geolocator.geocode(end_location + ", Melbourne", exactly_one=True)
        
        if not start_loc or not end_loc:
            return "Could not geocode one or both locations"
        
        # Find nearest stops to start and end
        start_stops = self.find_nearest_stops(start_loc.latitude, start_loc.longitude)
        end_stops = self.find_nearest_stops(end_loc.latitude, end_loc.longitude)
        
        if not start_stops or not end_stops:
            return "No transit stops found near one or both locations"
        
        # Find route
        path, transfers = self._find_multimodal_route(start_stops, end_stops, departure_time)
        
        if not path:
            return "No route found"
        
        # Add initial and final walking segments
        try:
            first_stop = self.stops[self.stops['stop_id'] == path[0]].iloc[0]
            last_stop = self.stops[self.stops['stop_id'] == path[-1]].iloc[0]
            
            # Initial walking segment
            initial_walk_time = self.get_walking_time(
                start_loc.latitude, start_loc.longitude,
                first_stop['stop_lat'], first_stop['stop_lon']
            )
            
            # Final walking segment
            final_walk_time = self.get_walking_time(
                last_stop['stop_lat'], last_stop['stop_lon'],
                end_loc.latitude, end_loc.longitude
            )
            
            # Get current time if not provided
            if departure_time is None:
                departure_time = datetime.now(pytz.timezone('Australia/Melbourne'))
            
            initial_arrival = departure_time + timedelta(seconds=initial_walk_time)
            final_departure = transfers[-1]['arrival_time'] if transfers else initial_arrival
            final_arrival = final_departure + timedelta(seconds=final_walk_time)
            
            # Add walking segments in consistent format
            transfers = [{
                'type': 'walking',
                'from_lat': start_loc.latitude,
                'from_lon': start_loc.longitude,
                'to_lat': first_stop['stop_lat'],
                'to_lon': first_stop['stop_lon'],
                'duration': initial_walk_time,
                'start_time': departure_time,
                'end_time': initial_arrival,
                'description': f'Walk to {first_stop["stop_name"]}'
            }] + transfers + [{
                'type': 'walking',
                'from_lat': last_stop['stop_lat'],
                'from_lon': last_stop['stop_lon'],
                'to_lat': end_loc.latitude,
                'to_lon': end_loc.longitude,
                'duration': final_walk_time,
                'start_time': final_departure,
                'end_time': final_arrival,
                'description': f'Walk from {last_stop["stop_name"]} to destination'
            }]
            
        except Exception as e:
            print(f"Warning: Could not generate walking segments: {e}")
        
        # Visualize route
        map_file = self.visualize_route(
            (start_loc.latitude, start_loc.longitude),
            (end_loc.latitude, end_loc.longitude),
            path,
            transfers
        )
        
        # Generate detailed text directions
        directions = ["Route found:"]
        total_walking = 0
        total_time = 0
        total_transfers = 0
        
        for i, transfer in enumerate(transfers):
            if transfer['type'] == 'walking':
                walk_time = transfer['duration']
                start_time = transfer['start_time'].strftime('%H:%M')
                end_time = transfer['end_time'].strftime('%H:%M')
                
                if 'description' in transfer:
                    # Initial or final walking segment
                    directions.append(f"{start_time} - {end_time}: {transfer['description']} ({walk_time/60:.0f} min walk)")
                else:
                    # Walking between stops
                    directions.append(f"{start_time} - {end_time}: Walk {walk_time/60:.0f} min to connect to next service")
                total_walking += walk_time
            else:
                # Transit segment
                departure = transfer['departure_time'].strftime('%H:%M')
                arrival = transfer['arrival_time'].strftime('%H:%M')
                wait_time = transfer['wait_time']
                transit_time = transfer['transit_time']
                
                to_stop = self.stops[self.stops['stop_id'] == transfer['to_stop']].iloc[0]
                
                # Add waiting time info if significant
                if wait_time > 60 and i > 0:  # Don't show initial wait
                    directions.append(f"{transfer['departure_time'].strftime('%H:%M')}: Wait {wait_time/60:.0f} min for next service")
                
                directions.append(
                    f"{departure} - {arrival}: Take {transfer['mode']} {transfer['route']} to {to_stop['stop_name']} "
                    f"({transit_time/60:.0f} min journey)"
                )
                
                total_time += wait_time + transit_time
                total_transfers += 1
        
        # Add summary statistics
        total_time += total_walking
        directions.append("\nJourney Summary:")
        directions.append(f"Departure: {transfers[0]['start_time'].strftime('%H:%M')}")
        directions.append(f"Arrival: {transfers[-1]['end_time'].strftime('%H:%M')}")
        directions.append(f"Total journey time: {total_time/60:.0f} minutes")
        directions.append(f"Total walking time: {total_walking/60:.0f} minutes")
        directions.append(f"Number of transfers: {total_transfers - 1}")  # Subtract 1 as first transit isn't a transfer
        
        return "\n".join(directions), map_file

if __name__ == "__main__":
    router = MultimodalTransitRouter()
    
    print("\nPlease enter your start and end locations to find a route.\n")
    start = input("Enter starting location: ")
    end = input("Enter destination: ")
    
    # Optionally specify a departure time
    use_custom_time = input("Use custom departure time? (y/n): ").lower().strip() == 'y'
    departure_time = None
    
    if use_custom_time:
        while True:
            try:
                time_str = input("Enter departure time (HH:MM): ")
                now = datetime.now(pytz.timezone('Australia/Melbourne'))
                hours, minutes = map(int, time_str.split(':'))
                departure_time = now.replace(hour=hours, minute=minutes, second=0, microsecond=0)
                break
            except ValueError:
                print("Invalid time format. Please use HH:MM (24-hour format)")
    
    result = router.find_route(start, end, departure_time)
    if isinstance(result, tuple):
        directions, map_file = result
        print("\nDirections:")
        print(directions)
        print("\nMap saved to:", map_file)
        webbrowser.open('file://' + os.path.abspath(map_file))
    else:
        print("\nError:", result)

Starting router initialization

Loading GTFS files
File loading took 2.57 seconds
Merging dataframes took 0.47 seconds
Building mappings took 1.01 seconds
Loading shape cache from disk
Cache loaded in 0.01 seconds

Total initialization took 4.06 seconds

Please enter your start and end locations to find a route.



Enter starting location:  Essendon
Enter destination:  Prahran
Use custom departure time? (y/n):  n


TypeError: unsupported type for timedelta seconds component: numpy.int64